# NYISO Extension: Zone J (NYC) and Zone K (Long Island)
Same pipeline as the PJM Western Hub model, applied to New York.

- **Primary Y:** daily on-peak day-ahead LBMP, NYISO Zone J (NYC), $/MWh
- **Secondary Y:** same for Zone K (Long Island)
- **On-peak:** weekdays, hours ending 8 to 23 Eastern, NERC holidays excluded (identical to PJM build)
- **X:** heating/cooling degree days (NYC for Zone J, Islip for Zone K), Henry Hub from the previous trading day, day-of-week and month fixed effects
- **Split:** train 2023 to 2025, test 2026 (chronological)

**Caveats:** (1) backtest uses observed weather, i.e. a perfect weather forecast; (2) Henry Hub understates NYC winter gas cost. Transco Zone 6 NY is the relevant gas price, but no free daily series exists (EIA licenses it from a vendor and publishes it only weekly). If you obtain one, save it as `transco_z6_ny.csv` (columns `date,gas`) and the notebook adds it as an alternative driver.

Run with **Runtime > Run all**. If any step fails, the notebook stops with a message instead of substituting values.

In [ ]:
# Step 1: install packages (Colab only)
import subprocess, sys, importlib.util
if importlib.util.find_spec("google.colab") is not None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gridstatusio", "xlrd"], check=False)

In [ ]:
# Step 2: settings
# Explain: one place for the study window, the train/test cutoff, and the weather stations.
import os, re, logging, warnings
import numpy as np
import pandas as pd
import requests
import statsmodels.formula.api as smf
from statsmodels.tsa.statespace.sarimax import SARIMAX
import matplotlib.pyplot as plt
from pandas.tseries.holiday import (AbstractHolidayCalendar, Holiday, USMemorialDay,
                                    USLaborDay, USThanksgivingDay, sunday_to_monday)

warnings.filterwarnings("ignore")
logging.getLogger("gridstatusio").setLevel(logging.WARNING)
pd.set_option("display.width", 160)

START = "2023-01-01"
END = (pd.Timestamp.today().normalize() - pd.Timedelta(days=2)).strftime("%Y-%m-%d")
TEST_START = "2026-01-01"
STATIONS = {"J": ("New York City", 40.71, -74.01), "K": ("Islip (Long Island)", 40.79, -73.10)}
NY_CACHE = "nyiso_daily.csv"
PJM_CACHE = "pjm_daily.csv"   # reused from the PJM notebook if present in this session

class StopRun(Exception):
    """Raised when a step fails, so nothing downstream runs on bad or missing data."""

print("Data window:", START, "to", END)

In [ ]:
# Step 3: shared functions
# Explain: the same on-peak block logic as PJM: convert UTC to Eastern, tag weekday hours
# ending 8 to 23 (interval start 7:00 to 22:00), drop NERC holidays, then average by day.
class NERCHolidays(AbstractHolidayCalendar):
    rules = [
        Holiday("New Year", month=1, day=1, observance=sunday_to_monday),
        USMemorialDay,
        Holiday("Independence Day", month=7, day=4, observance=sunday_to_monday),
        USLaborDay,
        USThanksgivingDay,
        Holiday("Christmas", month=12, day=25, observance=sunday_to_monday),
    ]

HOLIDAYS = NERCHolidays().holidays(start="2022-12-01", end="2027-12-31")

def get_client():
    from gridstatusio import GridStatusClient
    try:
        from google.colab import userdata
        key = userdata.get("GRIDSTATUS_API_KEY")
    except Exception:
        key = os.environ.get("GRIDSTATUS_API_KEY")
    return GridStatusClient(api_key=key)

def daily_blocks(hourly, prefix):
    t = pd.to_datetime(hourly["interval_start_utc"], utc=True).dt.tz_convert("America/New_York")
    d = pd.DataFrame({"t": t.values, "lmp": hourly["lmp"].values})
    d["t"] = pd.to_datetime(d["t"], utc=True).dt.tz_convert("America/New_York")
    d["date"] = d["t"].dt.tz_localize(None).dt.normalize()
    d["onpeak"] = ((d["t"].dt.dayofweek < 5) & d["t"].dt.hour.between(7, 22)
                   & ~d["date"].isin(HOLIDAYS))
    out = d.groupby(["date", "onpeak"])["lmp"].mean().unstack()
    return out.rename(columns={True: f"{prefix}_on", False: f"{prefix}_off"})

def get_weather(lat, lon, prefix):
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", params={
        "latitude": lat, "longitude": lon, "start_date": START, "end_date": END,
        "daily": "temperature_2m_mean", "temperature_unit": "fahrenheit",
        "timezone": "America/New_York"}, timeout=60)
    if r.status_code != 200:
        raise StopRun(f"Open-Meteo returned {r.status_code}: {r.text[:200]}")
    js = r.json()["daily"]
    temp = pd.Series(js["temperature_2m_mean"], index=pd.to_datetime(js["time"]))
    w = pd.DataFrame({f"{prefix}_temp": temp,
                      f"{prefix}_hdd": (65 - temp).clip(lower=0),
                      f"{prefix}_cdd": (temp - 65).clip(lower=0)})
    w.index.name = "date"
    return w

def get_henry_hub():
    try:
        raw = pd.read_excel("https://www.eia.gov/dnav/ng/hist_xls/RNGWHHDd.xls",
                            sheet_name="Data 1", skiprows=2)
        g = raw.iloc[:, :2]
        g.columns = ["date", "gas"]
    except Exception as e:
        if os.path.exists("henry_hub.csv"):
            print("EIA download failed; using henry_hub.csv")
            g = pd.read_csv("henry_hub.csv")
        else:
            raise StopRun(f"Henry Hub download failed ({e}). Upload henry_hub.csv with columns date,gas and rerun.")
    g["date"] = pd.to_datetime(g["date"])
    return g.dropna().sort_values("date")

def attach_prev_day_gas(daily, gas, col):
    """Gas known before the DA market closes: last price traded strictly before the delivery day."""
    daily = daily.copy()
    daily["lookup"] = daily["date"] - pd.Timedelta(days=1)
    merged = pd.merge_asof(daily.sort_values("lookup"),
                           gas.rename(columns={"date": "gas_date", "gas": col}),
                           left_on="lookup", right_on="gas_date", direction="backward",
                           tolerance=pd.Timedelta(days=7))
    return merged.drop(columns=["lookup", "gas_date"]).sort_values("date").reset_index(drop=True)

def show(df):
    try:
        print(df.to_markdown(index=False))
    except Exception:
        print(df.to_string(index=False))

In [ ]:
# Step 4: find the NYISO dataset and the Zone J / Zone K location names
# Explain: dataset IDs and zone spellings come from the API itself, not from guesses.
client = get_client()
ids = [d["id"] for d in client.list_datasets(return_list=True)]
nyiso_lmp = [i for i in ids if i.startswith("nyiso") and ("lmp" in i or "lbmp" in i)]
print("NYISO price datasets available:\n ", "\n  ".join(nyiso_lmp))

da_candidates = [i for i in nyiso_lmp if "day_ahead" in i and "hourly" in i]
if not da_candidates:
    raise StopRun("No NYISO day-ahead hourly price dataset found. Check the list above and set DATASET manually.")
DATASET = "nyiso_lmp_day_ahead_hourly" if "nyiso_lmp_day_ahead_hourly" in da_candidates else da_candidates[0]
print("\nUsing dataset:", DATASET)

probe = client.get_dataset(dataset=DATASET, start="2026-11-02", end="2026-11-02 01:00", limit=5000)
locs = probe[["location", "location_type"]].drop_duplicates() if "location_type" in probe else probe[["location"]].drop_duplicates()
zone_j = [l for l in locs["location"] if re.fullmatch(r"N\.?\s?Y\.?\s?C\.?", str(l).strip(), re.I)]
zone_k = [l for l in locs["location"] if re.fullmatch(r"LONG\s?IL.*|LONG ISLAND", str(l).strip(), re.I)]
print("\nZone-type locations in the probe:")
print(locs[locs.get("location_type", pd.Series("", index=locs.index)).astype(str).str.upper().str.contains("ZONE")].to_string(index=False)
      if "location_type" in locs else locs.head(30).to_string(index=False))
if not zone_j or not zone_k:
    raise StopRun(f"Could not match Zone J/K names automatically (found J={zone_j}, K={zone_k}). "
                  "Read the list above and set ZONE_NAMES manually.")
ZONE_NAMES = {"J": zone_j[0], "K": zone_k[0]}
print("\nZone names:", ZONE_NAMES)

In [ ]:
# Step 5: build or load the daily NYISO dataset
# Explain: pull hourly DA prices for each zone only (filtered server-side to save API rows),
# build on-peak/off-peak blocks, join zone weather and previous-day Henry Hub.
if os.path.exists(NY_CACHE):
    ny = pd.read_csv(NY_CACHE, parse_dates=["date"])
    print("Loaded cache:", NY_CACHE, ny.shape)
else:
    parts = []
    for z, name in ZONE_NAMES.items():
        hourly = client.get_dataset(dataset=DATASET, start=START, end=END,
                                    filter_column="location", filter_value=name)
        if hourly.empty:
            raise StopRun(f"No rows returned for {name}.")
        print(f"Zone {z} ({name}): {len(hourly):,} hourly rows")
        parts.append(daily_blocks(hourly[["interval_start_utc", "lmp"]], z))
    ny = parts[0].join(parts[1], how="outer")
    for z, (label, lat, lon) in STATIONS.items():
        ny = ny.join(get_weather(lat, lon, z), how="left")
    ny = ny.reset_index()
    ny = attach_prev_day_gas(ny, get_henry_hub(), "gas")
    if os.path.exists("transco_z6_ny.csv"):
        tz6 = pd.read_csv("transco_z6_ny.csv", parse_dates=["date"]).dropna().sort_values("date")
        ny = attach_prev_day_gas(ny, tz6, "gas_z6")
        print("Added Transco Z6 NY from transco_z6_ny.csv")
    ny.to_csv(NY_CACHE, index=False)
    print("Built and cached:", ny.shape)

print("\nMissing values:\n", ny.isna().sum())
ny.tail()

In [ ]:
# Step 6: model functions (identical method to the PJM build)
# Explain: every model forecasts a day using only information available before that day.
# Smoothing models use past prices; regressions use weather and previous-day gas.
def model_frame(daily, z, gas_col="gas"):
    df = daily[["date", f"{z}_on", f"{z}_temp", f"{z}_hdd", f"{z}_cdd", gas_col]].dropna().copy()
    df.columns = ["date", "da_on", "temp", "hdd", "cdd", "gas"]
    df = df.sort_values("date").reset_index(drop=True)
    df["dow"] = df["date"].dt.dayofweek
    df["month"] = df["date"].dt.month
    df["year"] = df["date"].dt.year
    df["heat_rate"] = df["da_on"] / df["gas"]
    df["lag1"] = df["da_on"].shift(1)
    df["last_year"] = df["date"].map(df.set_index("date")["da_on"].rename(lambda d: d + pd.Timedelta(days=364)))
    return df.dropna(subset=["lag1"]).reset_index(drop=True)

FORMULAS = {
    "Regression: weather + gas + FE": "da_on ~ hdd + cdd + gas + C(dow) + C(month)",
    "Regression: + squared weather": "da_on ~ hdd + I(hdd**2) + cdd + I(cdd**2) + gas + C(dow) + C(month)",
    "Regression: + squared weather + lag": "da_on ~ hdd + I(hdd**2) + cdd + I(cdd**2) + gas + lag1 + C(dow) + C(month)",
}

def run_models(df):
    y = df["da_on"]
    tr = df["date"] < TEST_START
    train, test = df[tr], df[~tr]
    preds = pd.DataFrame(index=df.index)
    preds["Naive: previous day"] = df["lag1"]
    preds["Naive: same day last year"] = df["last_year"]
    for w in [2, 5, 10, 20, 60]:
        preds[f"Moving average {w}d"] = y.shift(1).rolling(w).mean()
    for w in [5, 20]:
        wt = np.arange(1, w + 1)
        preds[f"Weighted MA {w}d"] = y.shift(1).rolling(w).apply(lambda x: np.dot(x, wt) / wt.sum(), raw=True)
    alpha = min(np.arange(0.05, 1.0, 0.05),
                key=lambda a: (y - y.ewm(alpha=a, adjust=False).mean().shift(1))[tr].abs().mean())
    preds[f"Exp. smoothing (alpha={alpha:.2f})"] = y.ewm(alpha=alpha, adjust=False).mean().shift(1)
    fits = {}
    for name, f in FORMULAS.items():
        fits[name] = smf.ols(f, data=train).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
        preds.loc[test.index, name] = fits[name].predict(test)
    hr = smf.ols("heat_rate ~ hdd + I(hdd**2) + cdd + I(cdd**2) + C(dow) + C(month)",
                 data=train).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    preds.loc[test.index, "Heat-rate model x gas"] = hr.predict(test) * test["gas"]
    try:
        def exog(d):
            X = d[["hdd", "cdd", "gas"]].copy()
            X["hdd2"] = d["hdd"] ** 2
            for k in range(1, 5):
                X[f"dow{k}"] = (d["dow"] == k).astype(float)
            return X.astype(float).values
        s_tr = SARIMAX(train["da_on"].values, exog=exog(train), order=(1, 0, 1), trend="c").fit(disp=False)
        s_all = SARIMAX(df["da_on"].values, exog=exog(df), order=(1, 0, 1), trend="c").filter(s_tr.params)
        preds.loc[test.index, "SARIMAX(1,0,1) + weather/gas"] = s_all.predict()[test.index]
    except Exception as e:
        print("SARIMAX skipped:", e)

    P = preds.loc[test.index].dropna()
    A = y.loc[P.index]
    winter = df.loc[P.index, "month"].isin([12, 1, 2])
    rows = []
    for m in P.columns:
        e = A - P[m]
        rows.append({"Model": m, "MAE": e.abs().mean(), "RMSE": np.sqrt((e ** 2).mean()),
                     "Bias (actual - pred)": e.mean(), "MAE winter": e[winter].abs().mean(),
                     "MAE rest": e[~winter].abs().mean(), "Worst miss": e.abs().max()})
    table = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True).round(2)
    base = fits["Regression: weather + gas + FE"]
    sens = pd.DataFrame({"Coefficient": base.params, "HAC std error": base.bse,
                         "p-value": base.pvalues}).loc[["hdd", "cdd", "gas"]].round(3)
    return {"table": table, "sens": sens, "preds": P, "actual": A, "df": df, "n_test": len(P)}

In [ ]:
# Step 7: run Zone J and Zone K
results = {}
for z in ["J", "K"]:
    df_z = model_frame(ny, z)
    print(f"\n=== Zone {z}: {len(df_z)} on-peak days "
          f"({df_z.date.min():%Y-%m-%d} to {df_z.date.max():%Y-%m-%d}) ===")
    results[z] = run_models(df_z)
    print(f"Out-of-sample comparison, {results[z]['n_test']} on-peak days in {TEST_START[:4]} ($/MWh)")
    show(results[z]["table"])
    results[z]["table"].to_csv(f"nyiso_zone_{z}_model_comparison.csv", index=False)

if "gas_z6" in ny.columns:
    print("\n=== Zone J with Transco Z6 NY as the gas driver ===")
    results["J_z6"] = run_models(model_frame(ny, "J", gas_col="gas_z6"))
    show(results["J_z6"]["table"].head(5))

In [ ]:
# Step 8: sensitivities (training period, 2023 to 2025)
# Explain: $/MWh change in the on-peak price per extra heating degree day, cooling degree day,
# and per $1/MMBtu of Henry Hub, holding day-of-week and month fixed effects constant.
for z in ["J", "K"]:
    print(f"\nZone {z} sensitivities ($/MWh on-peak):")
    print(results[z]["sens"])

In [ ]:
# Step 9: Zone J minus PJM Western Hub on-peak spread
# Explain: positive spread = NYC priced above PJM West. Reflects NYC's import constraints,
# local generation requirements, and gas basis. Uses the PJM daily file if present; otherwise pulls it.
if os.path.exists(PJM_CACHE):
    pjm = pd.read_csv(PJM_CACHE, parse_dates=["date"])[["date", "da_on"]]
    print("Using PJM daily data from", PJM_CACHE)
else:
    print("pjm_daily.csv not found; pulling PJM Western Hub DA hourly")
    pjm_h = client.get_dataset(dataset="pjm_lmp_day_ahead_hourly", start=START, end=END,
                               filter_column="location", filter_value="WESTERN HUB")
    pjm = daily_blocks(pjm_h[["interval_start_utc", "lmp"]], "da").reset_index()[["date", "da_on"]]

sp = ny[["date", "J_on"]].merge(pjm.rename(columns={"da_on": "pjm_on"}), on="date").dropna()
sp["spread"] = sp["J_on"] - sp["pjm_on"]
sp["year"] = sp["date"].dt.year
sp["month"] = sp["date"].dt.month

by_year = sp.groupby("year").agg(days=("spread", "size"), zone_j=("J_on", "mean"),
                                  pjm_west=("pjm_on", "mean"), avg_spread=("spread", "mean"),
                                  median_spread=("spread", "median")).round(2).reset_index()
by_month = sp.groupby("month").agg(avg_spread=("spread", "mean"), median_spread=("spread", "median"),
                                    days=("spread", "size")).round(2).reset_index()
widest = sp.reindex(sp["spread"].abs().sort_values(ascending=False).index).head(10)
widest = widest.assign(date=widest["date"].dt.strftime("%Y-%m-%d (%a)"))[["date", "J_on", "pjm_on", "spread"]].round(2)

print("Spread by year ($/MWh):"); show(by_year)
print("\nSpread by calendar month, all years ($/MWh):"); show(by_month)
print("\n10 widest spread days:"); show(widest)
for name, t in [("spread_by_year", by_year), ("spread_by_month", by_month), ("spread_widest_days", widest)]:
    t.to_csv(f"nyiso_{name}.csv", index=False)

In [ ]:
# Step 10: largest Zone J forecast misses (best model, 2026 out-of-sample)
# Explain: dates to research (heat waves, cold snaps, transmission or generation outages).
rJ = results["J"]
FUNDAMENTAL = ("Regression", "Heat-rate", "SARIMAX")
best_J = rJ["table"].loc[0, "Model"]                                   # best of all 15
best_fund = rJ["table"][rJ["table"]["Model"].str.startswith(FUNDAMENTAL)].iloc[0]["Model"]  # best weather/gas model
print("Best overall:", best_J, "| Best fundamentals model:", best_fund)
err = (rJ["actual"] - rJ["preds"][best_fund])
top = err.abs().sort_values(ascending=False).head(10).index
misses = rJ["df"].loc[top, ["date", "da_on", "temp", "hdd", "cdd", "gas"]].copy()
misses.insert(2, "forecast", rJ["preds"].loc[top, best_fund].values)
misses.insert(3, "miss", err.loc[top].values)
misses["date"] = misses["date"].dt.strftime("%Y-%m-%d (%a)")
print(f"Top 10 Zone J misses, model: {best_fund}"); show(misses.round(1))
misses.round(1).to_csv("nyiso_zone_J_largest_misses.csv", index=False)

In [ ]:
# Step 11: charts
fig, axes = plt.subplots(2, 1, figsize=(11, 8))
dJ = rJ["df"].loc[rJ["preds"].index, "date"]
axes[0].plot(dJ, rJ["actual"], color="black", lw=1.2, label="Actual Zone J DA on-peak")
axes[0].plot(dJ, rJ["preds"][best_fund], lw=1, label=f"Best fundamentals: {best_fund}")
axes[0].plot(dJ, rJ["preds"]["Naive: previous day"], lw=0.8, alpha=0.6, label="Naive: previous day")
axes[0].set_title(f"NYISO Zone J on-peak DA: actual vs forecast ({TEST_START[:4]}, out-of-sample)")
axes[0].set_ylabel("$/MWh"); axes[0].legend()
m = sp.set_index("date")["spread"].resample("MS").mean()
axes[1].bar(m.index, m.values, width=20)
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_title("Zone J minus PJM Western Hub, monthly average on-peak spread")
axes[1].set_ylabel("$/MWh")
plt.tight_layout(); plt.savefig("nyiso_results.png", dpi=150); plt.show()

In [ ]:
# Step 12: README section and resume line, filled from the results above
tJ = rJ["table"].set_index("Model")
mae_best = tJ.loc[best_fund, "MAE"]
mae_naive = tJ.loc["Naive: previous day", "MAE"]
beats_naive = mae_best < mae_naive
yr_last = by_year.iloc[-1]
widest_month = by_month.loc[by_month["avg_spread"].abs().idxmax()]
month_name = pd.Timestamp(2000, int(widest_month["month"]), 1).strftime("%B")
pct = (1 - mae_best / mae_naive) * 100
finding = (f"Zone J averaged ${yr_last['avg_spread']:+.2f}/MWh vs. PJM West in {int(yr_last['year'])}; "
           f"widest in {month_name} (avg ${widest_month['avg_spread']:+.2f}/MWh)")

readme = f"""## NYISO extension: Zone J (NYC) and Zone K (Long Island)

**Question:** do the weather-and-gas fundamentals that explain PJM Western Hub also explain New York City and Long Island on-peak power, and how does NYC price relative to PJM?

**Data:** NYISO hourly day-ahead zonal LBMPs from the Grid Status API (dataset `{DATASET}`, zones `{ZONE_NAMES['J']}` and `{ZONE_NAMES['K']}`), {START} to {END}; daily mean temperature for New York City (Zone J) and Islip (Zone K) from Open-Meteo, converted to degree days (base 65F); Henry Hub spot from EIA, previous trading day.

**Method:** identical to the PJM build: on-peak blocks (weekdays, HE8 to HE23 Eastern, NERC holidays excluded), chronological split (train 2023 to 2025, test 2026), 15-model out-of-sample comparison, HAC standard errors.

**Results (2026 out-of-sample, Zone J):** best weather/gas model {best_fund}, MAE ${mae_best:.2f}/MWh vs. ${mae_naive:.2f}/MWh for the naive previous-day benchmark ({abs(pct):.0f}% {"lower" if beats_naive else "higher"}). Best of all 15 models: {best_J} (MAE ${tJ.loc[best_J, 'MAE']:.2f}/MWh). Zone K best of all 15: {results['K']['table'].loc[0, 'Model']} (MAE ${results['K']['table'].loc[0, 'MAE']:.2f}/MWh).

**Zone J vs. PJM West:** {finding}.

**Limitations:** Henry Hub understates NYC gas cost in winter; Transco Zone 6 NY, the relevant local price, has no free daily series. Backtest uses observed weather (perfect-forecast assumption). No transmission, outage, or load-forecast variables yet.
"""
with open("README_NYISO_section.md", "w") as f:
    f.write(readme)
print(readme)
print("RESUME LINE (verify every number against the tables above):")
if beats_naive:
    print(f"Extended the model to NYISO Zone J (NYC): ${mae_best:.2f}/MWh MAE vs. ${mae_naive:.2f}/MWh naive; {finding}")
else:
    print(f"NOTE: the best weather/gas model ({best_fund}, ${mae_best:.2f}/MWh) did NOT beat naive (${mae_naive:.2f}/MWh) in Zone J. "
          "Do not use the standard resume line; report the finding instead, e.g.:")
    print(f"Extended the model to NYISO Zone J (NYC): price persistence beat weather/gas fundamentals out-of-sample; {finding}")